In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os
import pandas as pd

# Change the current working directory to your Google Drive
os.chdir('/content/drive/MyDrive')

# Go to Projects directory
os.chdir('Projects')

print("Files in Projects directory:", os.listdir())

# Open the CSV file
df = pd.read_csv('preprocessed_dataset.csv')

Files in Projects directory: ['preprocessed_dataset.csv']


In [4]:
# Ensure cleaned_text is string and handle NaNs
df['cleaned_text'] = df['cleaned_text'].astype(str).fillna("")

# Strip leading/trailing whitespace
df['cleaned_text'] = df['cleaned_text'].str.strip()

# Remove rows where cleaned_text is empty or 'nan'
df = df[df['cleaned_text'].str.lower() != 'nan']
df = df[df['cleaned_text'] != ""]

# Verify data types
print(df['cleaned_text'].apply(type).value_counts())  # Should show only <class 'str'>


cleaned_text
<class 'str'>    5567
Name: count, dtype: int64


In [17]:
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

# Improved normalization: preserve numeric tokens, normalize obfuscated chars inside words
def normalize_text(text):
    substitutions = {
        '0': 'o', '1': 'l', '3': 'e', '4': 'a', '@': 'a',
        '$': 's', '5': 's', '7': 't', '8': 'b', '®': 'r'
    }

    text = text.lower()
    tokens = text.split()

    normalized_tokens = []
    for token in tokens:
        if token.isdigit():
            # Keep pure numeric tokens as-is
            normalized_tokens.append(token)
        else:
            # Normalize characters inside words
            for key, val in substitutions.items():
                token = token.replace(key, val)
            # Remove unwanted characters except apostrophes and alphanumerics
            token = re.sub(r"[^a-z0-9']+", '', token)
            normalized_tokens.append(token)

    return " ".join(normalized_tokens)

# Custom tokenizer using the improved normalization
def custom_tokenizer(text):
    normalized = normalize_text(text)
    tokens = re.findall(r"[a-z0-9']+", normalized)
    return tokens

# Proper train/test split
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df['cleaned_text'], df['label'], test_size=0.2, random_state=123
)

# Optional: normalize text before vectorization (tokenizer also normalizes)
X_train_text = X_train_text.apply(normalize_text)
X_test_text = X_test_text.apply(normalize_text)

# Initialize TF-IDF vectorizer with custom tokenizer and n-grams
vectorizer = TfidfVectorizer(
    max_features=6000,
    tokenizer=custom_tokenizer,
    ngram_range=(1,2),
    analyzer='word'
)

# Fit on training data and transform train and test sets
X_train = vectorizer.fit_transform(X_train_text).toarray()
X_test = vectorizer.transform(X_test_text).toarray()

/usr/local/lib/python3.11/dist-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [18]:
from sklearn.naive_bayes import MultinomialNB

# Train Naive Bayes
nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)

# Now predict probabilities
train_probs = nb_model.predict_proba(X_train)
test_probs = nb_model.predict_proba(X_test)

In [19]:
from prettytable import PrettyTable

# === Test with new messages ===
new_texts = [
    "Congratulations! You won a free ticket to the FA Cup final. Text WIN to 12345 now!",
    "Hey, are we still meeting for lunch today?",
    "URGENT: Your account has been compromised, act now!",
    "Win $1000 by clicking this link now!",
    "Don't forget to submit your assignment tomorrow.",
    "Offer today, Buy now"
]

# Normalize and vectorize
new_texts_normalized = [normalize_text(t) for t in new_texts]
new_vectors = vectorizer.transform(new_texts_normalized)

# Predictions
probs = nb_model.predict_proba(new_vectors)
preds = nb_model.predict(new_vectors)

# Create table
table = PrettyTable()
table.field_names = ["Text", "Prediction", "P(Ham)", "P(Spam)"]

for text, prob, pred in zip(new_texts, probs, preds):
    label = "Spam" if pred == 1 else "Ham"
    table.add_row([text[:60] + ("..." if len(text) > 60 else ""), label, f"{prob[0]:.3f}", f"{prob[1]:.3f}"])

print(table)

+-----------------------------------------------------------------+------------+--------+---------+
|                               Text                              | Prediction | P(Ham) | P(Spam) |
+-----------------------------------------------------------------+------------+--------+---------+
| Congratulations! You won a free ticket to the FA Cup final. ... |    Spam    | 0.345  |  0.655  |
|            Hey, are we still meeting for lunch today?           |    Ham     | 0.995  |  0.005  |
|       URGENT: Your account has been compromised, act now!       |    Spam    | 0.468  |  0.532  |
|               Win $1000 by clicking this link now!              |    Spam    | 0.487  |  0.513  |
|         Don't forget to submit your assignment tomorrow.        |    Ham     | 0.957  |  0.043  |
|                       Offer today, Buy now                      |    Ham     | 0.953  |  0.047  |
+-----------------------------------------------------------------+------------+--------+---------+


In [20]:
# ======== Q-Learning ========
import numpy as np

num_states = 100
num_actions = 2  # 0 = Ham, 1 = Spam
Q = np.zeros((num_states, num_states, num_actions))

alpha = 0.3
gamma = 0.5
epsilon = 0.1  # Exploration rate after first encounter

# Track seen states
seen_states = set()

def discretize_state(prob):
    ham_prob = np.clip(prob[0], 0, 1)
    spam_prob = np.clip(prob[1], 0, 1)
    return int(ham_prob * (num_states - 1)), int(spam_prob * (num_states - 1))

def choose_action(state, prob=None):
    """
    First time: match NB prediction.
    Later: epsilon-greedy on Q-values.
    """
    if state not in seen_states and prob is not None:
        seen_states.add(state)
        return np.argmax(prob)  # Use NB prediction initially
    if np.random.rand() < epsilon:
        return np.random.choice(num_actions)
    return np.argmax(Q[state[0], state[1]])

def update_q_table(state, action, reward):
    """
    Standard Q-learning update.
    """
    current_q = Q[state[0], state[1], action]
    best_next_q = np.max(Q[state[0], state[1]])  # same state for simplicity
    Q[state[0], state[1], action] = current_q + alpha * (reward + gamma * best_next_q - current_q)

def get_user_selection():
    print("\nIs this correct?")
    print("1️⃣ Yes ✅")
    print("2️⃣ No ❌")
    while True:
        choice = input("Select option (1 or 2): ").strip()
        if choice == '1':
            return 'y'
        elif choice == '2':
            return 'n'
        else:
            print("Invalid choice. Please enter 1 or 2.")

# ======== Initialize Q-table from NB probabilities ========
train_probs = nb_model.predict_proba(X_train)
for prob in train_probs:
    state = discretize_state(prob)
    Q[state[0], state[1], 0] = prob[0]
    Q[state[0], state[1], 1] = prob[1]

# ======== Interactive Classification ========
print("\n🔍 Interactive Spam Classifier (type 'x' to quit) 🔍")
while True:
    user_msg = input("\nEnter a message (or 'x' to quit): ").strip()
    if user_msg.lower() == 'x':
        print("\n🚪 Exiting the classifier. Goodbye!")
        break

    # ✅ SAME normalizer and vectorizer as before
    normalized_msg = normalize_text(user_msg)
    X_user = vectorizer.transform([normalized_msg])
    prob = nb_model.predict_proba(X_user)[0]

    state = discretize_state(prob)
    action = choose_action(state, prob=prob)
    predicted_label = "Spam" if action == 1 else "Ham"
    q_values = Q[state[0], state[1]]
    emoji = "❌" if predicted_label == "Spam" else "✅"

    print(f"\nMessage: {user_msg}")
    print(f"Probabilities => Ham: {prob[0]:.3f}, Spam: {prob[1]:.3f}")
    print(f"Q-values => Ham: {q_values[0]:.4f}, Spam: {q_values[1]:.4f}")
    print("\n=== Prediction Result ===")
    print(f"{emoji} Predicted Label: {predicted_label}")

    correct = get_user_selection()
    if correct == 'y':
        update_q_table(state, action, 1)
    else:
        update_q_table(state, action, -1)
        correct_action = 1 - action
        update_q_table(state, correct_action, 1)

    print("🔄 Q-table updated based on your correction.")
    print("\n-------------------------------------------")



🔍 Interactive Spam Classifier (type 'x' to quit) 🔍

Enter a message (or 'x' to quit): Congratulations  i am pregnant send me money money 

Message: Congratulations  i am pregnant send me money money
Probabilities => Ham: 0.902, Spam: 0.098
Q-values => Ham: 0.9043, Spam: 0.0957

=== Prediction Result ===
✅ Predicted Label: Ham

Is this correct?
1️⃣ Yes ✅
2️⃣ No ❌
Select option (1 or 2): 1
🔄 Q-table updated based on your correction.

-------------------------------------------

Enter a message (or 'x' to quit): x

🚪 Exiting the classifier. Goodbye!
